# Webscraping

## Jobstreet.ph
- website has anti-scraping measures
- webscraping is against terms of service

url: https://ph.jobstreet.com/finance-and-admin-jobs 

In [11]:
"""
jobstreet_scraper.py
====================

This script scrapes job listings from the Jobstreet Philippines website for
finance and admin jobs.  It collects three pieces of information for each
unique company appearing in the search results:

1. **Company name** – extracted from each job card on the listing pages.
2. **Contact details** – e‑mail addresses or phone numbers appearing in the
   individual job advertisements.  Not all adverts publish contact details,
   so the list can be empty.
3. **Number of hirings** – the number of separate job adverts on Jobstreet
   associated with a company, counted across all processed pages.

Because Jobstreet uses infinite scroll and dynamic markup, the scraper relies
on HTML attributes that are present in the server‐rendered HTML.  In
particular, each job card includes an anchor tag containing the job title
(`data‑automation="jobTitle"`) and a nearby span with the company name
(`data‑automation="jobCompany"`).  The script walks through pages by
appending `?page=N` to the base search URL and stops when no new job cards
are found.  You can adjust the number of pages to crawl via the
``max_pages`` argument.

Usage:

    python jobstreet_scraper.py --pages 5 --output finance_admin_jobs.csv

The script writes a CSV file with columns ``company_name``, ``num_hirings``
and ``contact_details``.  Contact details are semicolon‑separated when
multiple values exist.

Note:
  - Running this scraper sends multiple HTTP requests to Jobstreet.  Be
    courteous by inserting a delay between requests (``delay`` argument).
  - Jobstreet does not always publish direct contact information; the
    ``contact_details`` field may therefore be empty for many companies.
"""

import csv
import re
import time
from collections import defaultdict
from typing import Dict, List, Set
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup


BASE_URL = "https://ph.jobstreet.com"
SEARCH_URL = "https://ph.jobstreet.com/finance-and-admin-jobs"


def extract_job_cards(html: str) -> List[dict]:
    """Parse a search results page and return a list of job entries.

    Each entry contains the job URL and company name.  The function looks for
    anchor tags with ``data‑automation="jobTitle"`` (job titles) and then
    finds the company name in the next ``span`` with
    ``data‑automation="jobCompany"`` within the same card.  This pattern
    reflects the HTML structure observed in Jobstreet's search results
    (see documentation lines around job cards)【960573467803511†L230-L232】.

    Args:
        html: Raw HTML of the search results page.

    Returns:
        A list of dictionaries with keys ``url`` and ``company``.
    """
    soup = BeautifulSoup(html, "html.parser")
    job_entries = []
    # Each job card has an anchor with this data attribute.
    for anchor in soup.find_all("a", attrs={"data-automation": "jobTitle"}):
        job_url = anchor.get("href")
        if not job_url:
            continue
        # Some links are relative; convert to absolute.
        job_link = urljoin(BASE_URL, job_url)
        # Find the company name associated with this job card.  The company
        # spans appear after the job title within the same card.  We search
        # the next elements in the DOM until we encounter a span with
        # ``data-automation`` set to ``jobCompany``.
        company_name = None
        next_elem = anchor
        while next_elem := next_elem.find_next():
            if getattr(next_elem, "name", None) == "span" and next_elem.get(
                "data-automation"
            ) == "jobCompany":
                company_name = next_elem.get_text(strip=True)
                break
            # Stop searching once we leave the card container (list item or
            # article).  This prevents capturing data from unrelated cards.
            if next_elem.name in {"li", "article"} and next_elem != anchor:
                break
        job_entries.append({"url": job_link, "company": company_name})
    return job_entries


def extract_contact_details(job_html: str) -> Set[str]:
    """Extract contact details (emails and phone numbers) from a job page.

    The Jobstreet job pages do not explicitly list recruiter contact info in
    structured fields, but sometimes the job description contains an email
    address or phone number.  This function searches the entire HTML for
    patterns matching e‑mails and Philippine phone numbers.  Multiple contact
    details are de‑duplicated using a set.

    Args:
        job_html: Raw HTML of the individual job page.

    Returns:
        A set of strings representing contact details.  Returns an empty set
        when no details are found.
    """
    contacts: Set[str] = set()
    # Simple e‑mail regex: username@domain.tld
    email_pattern = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
    # Philippine phone numbers may include country code +63 or local 0,
    # parentheses, spaces or dashes.  This regex captures sequences of at
    # least 10 digits and allows separators.
    phone_pattern = re.compile(
        r"(?:\+?\d{1,3}[\s\-]?)?(?:\(\d+\)[\s\-]?)?\d{3,4}[\s\-]?\d{3,4}"
    )
    for match in email_pattern.findall(job_html):
        contacts.add(match)
    for match in phone_pattern.findall(job_html):
        # Filter out numbers that are clearly not phone numbers (e.g., salary
        # ranges like 70,000 – 80,000).  Require at least 10 digits.
        digits = re.sub(r"\D", "", match)
        if len(digits) >= 10:
            contacts.add(match)
    return contacts


def scrape_jobstreet(pages: int = 1, delay: float = 1.0) -> Dict[str, Dict[str, List[str]]]:
    """Scrape finance/admin job listings and aggregate information by company.

    Args:
        pages: Number of pages to crawl.  The search results contain many
            pages; adjust this based on how comprehensive you want the crawl to be.
        delay: Seconds to pause between HTTP requests to be polite.

    Returns:
        A dictionary keyed by company name.  Each value is another dict with
        keys ``links`` (list of job URLs) and ``contacts`` (set of contact
        details found across all associated job pages).
    """
    session = requests.Session()
    # Imitate a browser to avoid basic bot detection.
    session.headers.update(
        {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/119.0.0.0 Safari/537.36"
            )
        }
    )
    companies: Dict[str, Dict[str, Set[str]]] = defaultdict(
        lambda: {"links": [], "contacts": set()}
    )
    for page_num in range(1, pages + 1):
        search_url = SEARCH_URL
        if page_num > 1:
            search_url = f"{SEARCH_URL}?page={page_num}"
        response = session.get(search_url, timeout=20)
        if response.status_code != 200:
            # Stop crawling if the server returns an error.
            break
        job_entries = extract_job_cards(response.text)
        if not job_entries:
            # No more jobs found; break early.
            break
        for entry in job_entries:
            company = entry["company"] or "Unknown"
            link = entry["url"]
            companies[company]["links"].append(link)
        # Be polite – avoid hammering the site.
        time.sleep(delay)
    # Fetch each job page and extract contact information.
    for company, data in companies.items():
        for link in data["links"]:
            try:
                resp = session.get(link, timeout=20)
            except Exception:
                continue
            if resp.status_code == 200:
                contacts = extract_contact_details(resp.text)
                data["contacts"].update(contacts)
            time.sleep(delay)
    return companies


def save_to_csv(companies: Dict[str, Dict[str, Set[str]]], filename: str) -> None:
    """Write the aggregated company information to a CSV file.

    Args:
        companies: Dictionary keyed by company names with link and contact info.
        filename: Path to the output CSV file.
    """
    with open(filename, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["company_name", "num_hirings", "contact_details"])
        for company, info in companies.items():
            num_hirings = len(info["links"])
            contacts = "; ".join(sorted(info["contacts"])) if info["contacts"] else ""
            writer.writerow([company, num_hirings, contacts])


def main():
    import argparse

    parser = argparse.ArgumentParser(
        description="Scrape Jobstreet finance/admin jobs and aggregate by company."
    )
    parser.add_argument(
        "--pages",
        type=int,
        default=1,
        help="Number of search result pages to crawl (default: 1)",
    )
    parser.add_argument(
        "--delay",
        type=float,
        default=1.0,
        help="Seconds to wait between HTTP requests (default: 1.0)",
    )
    parser.add_argument(
        "--output",
        type=str,
        default="jobstreet_finance_admin.csv",
        help="Filename for the output CSV (default: jobstreet_finance_admin.csv)",
    )
    args, _ = parser.parse_known_args()
    companies = scrape_jobstreet(pages=args.pages, delay=args.delay)
    save_to_csv(companies, args.output)
    print(
        f"Scraped {len(companies)} companies across {args.pages} page(s). Results saved to {args.output}."
    )


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'requests'

## PhilJobNet

In [9]:
"""
PhilJobNet Job Scraper - UPDATED for Real HTML Structure
Scrapes Business, Finance, & Admin job postings from PhilJobNet
Filters for companies hiring more than 1 person
Exports to Excel
"""

import asyncio
import pandas as pd
from playwright.async_api import async_playwright
from datetime import datetime
import re

class PhilJobNetScraper:
    def __init__(self):
        self.base_url = "https://philjobnet.gov.ph"
        self.jobs_url = "https://philjobnet.gov.ph/job-vacancies/"
        self.jobs_data = []
        
    async def scrape(self, max_pages=10):
        """Main scraping function"""
        async with async_playwright() as p:
            # Launch browser
            browser = await p.chromium.launch(headless=False)  # Set to True for background
            page = await browser.new_page()
            
            print("🚀 Starting PhilJobNet scraper...")
            print(f"Target: Business, Finance, & Admin roles")
            print(f"Filter: Companies hiring >1 person\n")
            
            try:
                # Navigate to job vacancies page
                await page.goto(self.jobs_url, wait_until="networkidle")
                await asyncio.sleep(2)
                
                print("📋 Loading job listings...\n")
                
                current_page = 1
                
                while current_page <= max_pages:
                    print(f"\n{'='*60}")
                    print(f"📄 PROCESSING PAGE {current_page}")
                    print(f"{'='*60}\n")
                    
                    # Extract job listings from current page
                    await self.extract_jobs_from_page(page)
                    
                    # Try to go to next page
                    try:
                        # Look for page 2, 3, 4, etc. links
                        next_page_num = current_page + 1
                        next_page_link = page.locator(f'a:has-text("{next_page_num}")')
                        
                        if await next_page_link.count() > 0:
                            print(f"\n⏭️  Moving to page {next_page_num}...")
                            await next_page_link.first.click()
                            await asyncio.sleep(3)  # Wait for page to load
                            current_page += 1
                        else:
                            print("\n✅ No more pages found. Scraping complete.")
                            break
                    except Exception as e:
                        print(f"\n✅ Reached last page or error in pagination: {e}")
                        break
                
                print(f"\n{'='*60}")
                print(f"✅ SCRAPING COMPLETE!")
                print(f"{'='*60}")
                print(f"📊 Total relevant jobs found: {len(self.jobs_data)}")
                
            except Exception as e:
                print(f"❌ Error during scraping: {e}")
            
            finally:
                await browser.close()
    
    async def extract_jobs_from_page(self, page):
        """Extract job listings from current page"""
        try:
            # Wait for job listings table to load
            await page.wait_for_selector('#ctl00_BodyContentPlaceHolder_GridView1', timeout=10000)
            
            # Get all job cards
            job_cards = page.locator('.jobcard')
            job_count = await job_cards.count()
            
            print(f"Found {job_count} job listings on this page\n")
            
            for i in range(job_count):
                try:
                    job_card = job_cards.nth(i)
                    
                    # Extract job title
                    job_title_elem = job_card.locator('.jobtitle')
                    job_title = await job_title_elem.inner_text() if await job_title_elem.count() > 0 else "N/A"
                    
                    # Extract company name
                    company_elem = job_card.locator('.companytitle')
                    company_name = await company_elem.inner_text() if await company_elem.count() > 0 else "N/A"
                    
                    # Extract job URL
                    job_link = job_card.locator('xpath=ancestor::a')
                    job_url = await job_link.get_attribute('href') if await job_link.count() > 0 else ""
                    
                    if job_url:
                        full_url = f"{self.base_url}{job_url}" if job_url.startswith('/') else job_url
                        
                        # Check if it's a Business/Finance/Admin job
                        if self.is_bfa_job(job_title):
                            print(f"  🔍 Checking: {job_title} at {company_name}")
                            await self.extract_job_details(page, full_url, job_title, company_name)
                        else:
                            print(f"  ⏭️  Skipped (not BFA): {job_title}")
                    
                except Exception as e:
                    print(f"  ⚠️  Error processing job {i+1}: {e}")
                    continue
                    
        except Exception as e:
            print(f"  ❌ Error extracting jobs from page: {e}")
    
    def is_bfa_job(self, job_title):
        """Check if job is Business/Finance/Admin related"""
        bfa_keywords = [
            'accounting', 'accountant', 'finance', 'financial', 'admin', 'administrative',
            'business', 'hr', 'human resource', 'payroll', 'bookkeeping', 'bookkeeper',
            'auditor', 'audit', 'analyst', 'cashier', 'billing', 'budget', 'tax',
            'controller', 'secretary', 'clerk', 'officer', 'assistant', 'coordinator',
            'manager', 'supervisor', 'executive', 'director', 'chief', 'head'
        ]
        
        job_title_lower = job_title.lower()
        return any(keyword in job_title_lower for keyword in bfa_keywords)
    
    async def extract_job_details(self, original_page, job_url, job_title, company_name):
        """Visit individual job page and extract details"""
        job_page = None
        try:
            # Open job detail page in new tab
            job_page = await original_page.context.new_page()
            await job_page.goto(job_url, wait_until="networkidle", timeout=30000)
            await asyncio.sleep(2)
            
            # Get the full page content
            page_content = await job_page.content()
            page_text = await job_page.inner_text('body')
            
            # Initialize job data
            job_data = {
                'job_title': job_title,
                'company_name': company_name,
                'email': '',
                'contact_number': '',
                'number_of_hirings': '',
                'location': '',
                'salary': '',
                'job_type': '',
                'posted_date': '',
                'job_url': job_url
            }
            
            # Extract number of vacancies - look for common patterns
            vacancy_patterns = [
                r'number of vacancies?[:\s]+(\d+)',
                r'vacancies?[:\s]+(\d+)',
                r'(\d+)\s+vacanc(?:y|ies)',
                r'(\d+)\s+opening[s]?',
                r'(\d+)\s+position[s]?\s+available',
                r'hiring[:\s]+(\d+)',
                r'slots?[:\s]+(\d+)'
            ]
            
            for pattern in vacancy_patterns:
                match = re.search(pattern, page_text, re.IGNORECASE)
                if match:
                    job_data['number_of_hirings'] = match.group(1)
                    break
            
            # Extract email
            email_match = re.search(r'[\w\.-]+@[\w\.-]+\.\w+', page_text)
            if email_match:
                job_data['email'] = email_match.group(0)
            
            # Extract contact number - Philippine number patterns
            phone_patterns = [
                r'(\+63|0)\s*\d{3}\s*\d{3}\s*\d{4}',
                r'(\+63|0)\d{10}',
                r'\(0\d{2}\)\s*\d{3}[-\s]?\d{4}',
                r'0\d{2}[-\s]\d{3}[-\s]\d{4}'
            ]
            
            for pattern in phone_patterns:
                match = re.search(pattern, page_text)
                if match:
                    job_data['contact_number'] = match.group(0)
                    break
            
            # Extract location from page
            location_elem = job_page.locator('text=/location|workplace|address/i').first
            if await location_elem.count() > 0:
                location_text = await location_elem.inner_text()
                job_data['location'] = location_text[:100]
            
            # Extract salary
            salary_elem = job_page.locator('text=/salary|compensation|rate/i').first
            if await salary_elem.count() > 0:
                salary_text = await salary_elem.inner_text()
                job_data['salary'] = salary_text[:50]
            
            # Filter: Only add if hiring more than 1 person
            if job_data['number_of_hirings']:
                try:
                    num_hirings = int(job_data['number_of_hirings'])
                    if num_hirings > 1:
                        self.jobs_data.append(job_data)
                        print(f"    ✅ ADDED: {num_hirings} positions | Email: {job_data['email'] or 'Not found'} | Phone: {job_data['contact_number'] or 'Not found'}")
                    else:
                        print(f"    ⏭️  Skipped: Only 1 position")
                except ValueError:
                    print(f"    ⚠️  Could not parse vacancy count: {job_data['number_of_hirings']}")
            else:
                print(f"    ⚠️  No vacancy count found - skipping")
            
        except Exception as e:
            print(f"    ❌ Error extracting job details: {e}")
        finally:
            if job_page:
                await job_page.close()
    
    def export_to_excel(self, filename='philjobnet_jobs.xlsx'):
        """Export scraped data to Excel"""
        if not self.jobs_data:
            print("\n⚠️  No data to export!")
            return
        
        df = pd.DataFrame(self.jobs_data)
        
        # Reorder columns
        column_order = [
            'company_name',
            'email', 
            'contact_number',
            'number_of_hirings',
            'job_title',
            'location',
            'salary',
            'job_type',
            'posted_date',
            'job_url'
        ]
        
        df = df[column_order]
        
        # Export to Excel
        df.to_excel(filename, index=False, engine='openpyxl')
        
        print(f"\n{'='*60}")
        print(f"💾 DATA EXPORTED TO: {filename}")
        print(f"{'='*60}")
        print(f"📊 Total companies found: {len(df)}")
        
        # Print summary
        if len(df) > 0:
            print(f"\n📈 SUMMARY:")
            print(f"   - Total jobs matching criteria: {len(df)}")
            total_positions = df['number_of_hirings'].astype(int).sum()
            print(f"   - Total positions available: {total_positions}")
            with_email = df['email'].notna().sum()
            with_phone = df['contact_number'].notna().sum()
            print(f"   - Jobs with email: {with_email} ({with_email/len(df)*100:.1f}%)")
            print(f"   - Jobs with phone: {with_phone} ({with_phone/len(df)*100:.1f}%)")
            with_both = ((df['email'].notna()) & (df['contact_number'].notna())).sum()
            print(f"   - Jobs with both email & phone: {with_both} ({with_both/len(df)*100:.1f}%)")


async def main():
    scraper = PhilJobNetScraper()
    
    # Scrape up to 10 pages (adjust as needed)
    await scraper.scrape(max_pages=10)
    
    # Export to Excel
    scraper.export_to_excel('philjobnet_business_finance_admin_jobs.xlsx')


print("""
╔════════════════════════════════════════════════╗
║     PhilJobNet Job Scraper - UPDATED           ║
║     Business, Finance & Admin Focus            ║
║     Optimized for Real Website Structure       ║
╚════════════════════════════════════════════════╝
""")

if __name__ == "__main__":
    print("""
    ╔════════════════════════════════════════════════╗
    ║     PhilJobNet Job Scraper - UPDATED           ║
    ║     Business, Finance & Admin Focus            ║
    ║     Optimized for Real Website Structure       ║
    ╚════════════════════════════════════════════════╝
    """)
    
    asyncio.run(main())




╔════════════════════════════════════════════════╗
║     PhilJobNet Job Scraper - UPDATED           ║
║     Business, Finance & Admin Focus            ║
║     Optimized for Real Website Structure       ║
╚════════════════════════════════════════════════╝


    ╔════════════════════════════════════════════════╗
    ║     PhilJobNet Job Scraper - UPDATED           ║
    ║     Business, Finance & Admin Focus            ║
    ║     Optimized for Real Website Structure       ║
    ╚════════════════════════════════════════════════╝
    


RuntimeError: asyncio.run() cannot be called from a running event loop

# Kalibrr

In [10]:
import asyncio
asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

from playwright.async_api import async_playwright
import pandas as pd

async def scrape_kalibrr(keyword="accounting", pages=3):
    all_jobs = []
    base_url = "https://www.kalibrr.com"

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        for page_num in range(1, pages + 1):
            url = f"{base_url}/jobs?search={keyword}&page={page_num}"
            await page.goto(url, wait_until="networkidle")
            await page.wait_for_timeout(2000)

            cards = await page.query_selector_all("div[data-testid='job-card']")
            print(f"Found {len(cards)} cards on page {page_num}")

            for card in cards:
                title = await card.query_selector("h2")
                company = await card.query_selector("a[data-testid='company-name']")
                link = await card.query_selector("a[data-testid='job-link']")

                job = {
                    "job_title": await title.inner_text() if title else "",
                    "company_name": await company.inner_text() if company else "",
                    "job_url": await link.get_attribute("href") if link else "",
                }
                all_jobs.append(job)

        await browser.close()

    return pd.DataFrame(all_jobs)

df = await scrape_kalibrr("finance", pages=5)
df.to_csv("kalibrr_playwright.csv", index=False)
df.head()

NotImplementedError: 